In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from ECOv002_calval_tables import load_calval_table
from BESS_JPL import load_ECOv002_static_tower_BESS_inputs
from BESS_JPL import process_BESS_table

In [3]:
repo_root = os.path.dirname(os.getcwd())
package_dir = os.path.join(repo_root, 'BESS_JPL')
generated_input_table_filename = os.path.join(package_dir, "ECOv002-cal-val-BESS-JPL-inputs.csv")
generated_output_table_filename = os.path.join(package_dir, "ECOv002-cal-val-BESS-JPL-outputs.csv")

In [4]:
model_inputs_gdf = load_calval_table()
model_inputs_gdf["elevation_km"] = model_inputs_gdf["Elev"] / 1000.0
model_inputs_gdf.head()

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,emissivity,insitu_LE_Wm2,insitu_H_Wm2,insitu_Rn_Wm2,insitu_G_Wm2,insitu_SWin_Wm2,insitu_Rn_daylight_Wm2,insitu_LE_daylight_Wm2,insitu_ET_daylight_kg,elevation_km
0,0,US-NC3,ENF,Cfa,270.34520,78.53355,392.85184,307.02197,487.383423,118.91628,...,0.948,331.723100,69.623470,449.65123,14.831077,596.8641,268.829847,205.089552,3.521847,0.005
1,1,US-Mi3,CVM,Dfb,232.14160,229.20093,640.11847,375.08930,106.825577,167.91946,...,0.952,276.182074,228.756306,667.81900,38.638200,NaN,346.382557,152.046364,3.361398,0.270
2,2,US-Mi3,CVM,Dfb,356.35574,335.23154,625.66170,284.68625,NaN,132.93634,...,0.972,336.849808,169.322709,603.26000,26.775450,NaN,311.373843,181.941076,4.018909,0.270
3,3,US-Mi3,CVM,Dfb,332.93840,326.68680,624.25433,251.41449,178.827545,141.13242,...,0.974,276.264202,231.754374,551.03800,13.319750,NaN,298.014172,153.111127,3.377459,0.270
4,4,US-Mi3,CVM,Dfb,286.85403,237.21654,511.08218,228.52017,154.791626,114.80941,...,0.960,192.205091,155.974781,512.27100,6.896745,NaN,300.483101,114.280419,2.519399,0.270


In [5]:
static_inputs_df = load_ECOv002_static_tower_BESS_inputs()
static_inputs_df

,ID,name,NDVI_minimum,NDVI_maximum,C4_fraction,carbon_uptake_efficiency,kn,peakVCmax_C3,peakVCmax_C4,ball_berry_slope_C3,...,KG_climate,CI,canopy_height_meters,COT,AOT,Ca,wind_speed_mps,vapor_gccm,ozone_cm,geometry
0,US-NC3,NC_Clearcut#3,0.408733,0.855693,0.077532,0.080000,0.410000,87.345433,51.040624,9.5,...,3,0.282353,20.642902,0,0,400,0,0,0.3,POINT (-76.656 35.799)
1,PE-QFR,Quistococha Forest Reserve,0.657359,0.826605,0.000549,0.060347,0.125033,41.364487,41.364487,9.5,...,1,0.254902,22.140021,0,0,400,0,0,0.3,POINT (-73.319 -3.8344)
2,US-Mi3,LTAR UCB (Upper Chesapeake Bay) Miscanthus 3,0.027910,0.855461,0.035045,0.080000,0.410000,119.435443,119.435443,7.5,...,4,0.286275,0.000000,0,0,400,0,0,0.3,POINT (-80.637 41.8222)
3,US-NC4,NC_AlligatorRiver,0.591358,0.869758,0.046219,0.080000,0.410000,64.720165,64.720165,9.5,...,3,0.207843,14.164827,0,0,400,0,0,0.3,POINT (-75.9038 35.7879)
4,CA-DB2,Delta Burns Bog 2,0.399712,0.675731,0.000356,0.080000,0.410000,109.986395,109.986395,9.5,...,3,0.266667,9.919029,0,0,400,0,0,0.3,POINT (-122.9951 49.119)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,US-xSL,"NEON North Sterling, CO (STER)",-0.025449,0.495206,0.340776,0.090000,0.710000,78.000000,40.000000,9.5,...,2,0.298039,0.000000,0,0,400,0,0,0.3,POINT (-103.0293 40.4619)
117,US-xWD,NEON Woodworth (WOOD),-0.041785,0.753863,0.032479,0.080000,0.410000,101.000000,37.000000,7.5,...,4,0.294118,0.000000,0,0,400,0,0,0.3,POINT (-99.2414 47.1282)
118,US-CS4,Central Sands Irrigated Agricultural Field,-0.003026,0.776740,0.092454,0.080000,0.410000,101.000000,37.000000,7.5,...,4,0.278431,0.000000,0,0,400,0,0,0.3,POINT (-89.5475 44.1597)
119,US-xAE,NEON Klemme Range Research Station (OAES),0.233503,0.554538,0.371127,0.090000,0.710000,78.000000,40.000000,9.5,...,3,0.301961,0.000000,0,0,400,0,0,0.3,POINT (-99.0588 35.4106)


In [6]:
# merge static inputs with model inputs, ignoring duplicate columns from static_inputs_df
cols_to_use = [col for col in static_inputs_df.columns if col not in model_inputs_gdf.columns or col == 'ID']
model_inputs_gdf = model_inputs_gdf.merge(static_inputs_df[cols_to_use], on="ID", how="left")
model_inputs_gdf

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,ball_berry_intercept_C3,KG_climate,CI,canopy_height_meters,COT,AOT,Ca,wind_speed_mps,vapor_gccm,ozone_cm
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0.005267,3,0.282353,20.642902,0,0,400,0,0,0.3
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3


In [7]:
model_inputs_gdf.columns

Index(['Unnamed: 0', 'ID', 'vegetation', 'climate', 'STICinst', 'BESSinst',
       'MOD16inst', 'PTJPLSMinst', 'ETinst', 'ETinstUncertainty',
       ...
       'ball_berry_intercept_C3', 'KG_climate', 'CI', 'canopy_height_meters',
       'COT', 'AOT', 'Ca', 'wind_speed_mps', 'vapor_gccm', 'ozone_cm'],
      dtype='object', length=107)

In [8]:
results = process_BESS_table(model_inputs_gdf)
results

[2025-12-11 08:58:08 INFO] started extracting geometry from PT-JPL-SM input table
[2025-12-11 08:58:08 INFO] completed extracting geometry from PT-JPL-SM input table
[2025-12-11 08:58:08 INFO] started extracting time from PT-JPL-SM input table
[2025-12-11 08:58:08 INFO] completed extracting time from PT-JPL-SM input table
[2025-12-11 08:58:08 INFO] variable elevation_m min: 1.000 mean: 992.883 max: 3504.000 nan: 0.00% (nan)
[2025-12-11 08:58:08 INFO] variable Ta_C min: -14.605 mean: 22.322 max: 39.710 nan: 0.00% (nan)
[2025-12-11 08:58:08 INFO] variable RH min: 0.273 mean: 0.427 max: 0.984 nan: 0.00% (nan)
[2025-12-11 08:58:08 INFO] variable NDVI_minimum min: -0.033 mean: 0.174 max: 0.591 nan: 0.00% (nan)
[2025-12-11 08:58:08 INFO] variable NDVI_maximum min: 0.314 mean: 0.628 max: 0.918 nan: 0.00% (nan)
[2025-12-11 08:58:08 INFO] variable C4_fraction min: 0.000 mean: 0.296 max: 0.939 nan: 0.00% (nan)
[2025-12-11 08:58:08 INFO] variable carbon_uptake_efficiency min: 0.080 mean: 0.083 ma

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,ozone_cm,GPP,GPP_daily,Rn_Wm2,Rn_soil_Wm2,Rn_canopy_Wm2,LE_Wm2,LE_soil_Wm2,LE_canopy_Wm2,G_Wm2
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0.3,16.306166,5.963702,543.574452,343.088149,200.486302,253.346605,69.342530,184.004076,37.045927
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0.3,21.081705,8.623039,745.548466,568.929479,176.618987,225.765308,100.628344,125.136964,75.785987
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0.3,19.984936,8.120491,797.080090,595.207599,201.872491,346.807854,210.893347,135.914507,77.675725
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0.3,24.216486,10.357271,734.801464,539.752166,195.049298,336.703409,207.202546,129.500864,69.717554
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0.3,21.872920,10.242348,600.127482,441.941745,158.185737,259.677885,140.032416,119.645470,57.794581
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0.3,0.492826,0.193133,235.343662,165.684027,69.659635,46.681261,41.487778,5.193483,29.244792
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0.3,1.259612,0.882347,266.190406,190.262171,75.928235,42.918705,31.172900,11.745805,32.896579
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0.3,3.360783,2.404805,405.659580,266.716379,138.943201,47.368634,6.239409,41.129224,35.811898
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0.3,1.986428,1.197719,314.897472,247.995142,66.902329,59.134527,48.996894,10.137633,44.868316


In [9]:
model_inputs_gdf.to_csv(generated_input_table_filename, index=False)

In [10]:
results.to_csv(generated_output_table_filename, index=False)